# Chapter 6. Ensemble Learning and Random Forests

In Chapter 5, we saw that decision trees are powerful and interpretable models, but they suffer from high variance. A small change in the training data can produce a completely different tree structure.

Ensemble learning addresses this limitation by combining multiple predictors instead of relying on a single model.

The fundamental philosophy behind ensemble learning is the wisdom of the crowd. A group of diverse models can often produce more accurate and robust predictions than the best individual model.

An ensemble is especially powerful when the individual predictors make different types of errors. If their errors are not perfectly correlated, aggregating their predictions can reduce overall variance and improve generalization.

## Voting Classifiers

A voting classifier combines the predictions of multiple different models.

Instead of selecting one best model, we train several baseline classifiers and aggregate their predictions. This works best when the models are structurally diverse because different algorithms tend to make different kinds of mistakes.

For example, we can combine:
Logistic Regression, Random Forest, and Support Vector Machine

- Hard Voting: The ensemble predicts the class that receives the majority vote.

- Soft Voting: The ensemble averages the predicted class probabilities and selects the class with the highest average probability.

Soft voting often performs better than hard voting because it uses confidence information rather than only counting class labels.

In [4]:
from sklearn.datasets import make_moons
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

# Generate and split a nonlinear toy dataset
X, y = make_moons(n_samples=500, noise=0.30, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# Train Hard Voting Classifier
voting_clf = VotingClassifier(
    estimators=[
        ('lr', LogisticRegression(random_state=42)),
        ('rf', RandomForestClassifier(random_state=42)),
        ('svc', SVC(random_state=42))
    ]
)
voting_clf.fit(X_train, y_train)

# Evaluate individual baseline models
print('Individual accuracy')
for name, clf in voting_clf.named_estimators_.items():
    print(name, '=', clf.score(X_test, y_test))

Individual accuracy
lr = 0.864
rf = 0.896
svc = 0.896


In [5]:
# Show hard voting mechanism for the first instance
print('Ensemble Prediction \n', voting_clf.predict(X_test[:1]))
print('Individual Predictions \n', [clf.predict(X_test[:1]) for clf in voting_clf.estimators_])

Ensemble Prediction 
 [1]
Individual Predictions 
 [array([1], dtype=int64), array([1], dtype=int64), array([0], dtype=int64)]


In [6]:
# Evaluate Hard Voting Ensemble
print('Hard Voting Accuracy \n', voting_clf.score(X_test, y_test))

Hard Voting Accuracy 
 0.912


The voting classifier predicts class 1 for the first instance of the test set, because two out of three classifiers predict that class.

Now let's try Soft Voting to see if using probability estimates improves our accuracy. Since Soft Voting relies on confidence levels rather than just a strict majority count we first need to explicitly enable probability calculations for our SVM classifier.

In [8]:
# Upgrade to Soft Voting dynamically
voting_clf.voting = 'soft'
voting_clf.named_estimators['svc'].probability = True
voting_clf.fit(X_train, y_train)

# Evaluate Soft Voting Ensemble
print('Soft Voting Accuracy \n', voting_clf.score(X_test, y_test))

Soft Voting Accuracy 
 0.92


## Bagging and Pasting

Instead of utilizing different algorithms we can use the exact same training algorithm but train each independent predictor on a randomly sampled subset of the training data. This architecture allows models to be trained entirely in parallel across multiple CPU cores making it highly scalable.

* **Bagging:** Sampling training instances with replacement. This introduces more diversity resulting in a slightly higher bias but significantly lower variance. Bagging generally yields superior generalized models.
* **Pasting:** Sampling training instances strictly without replacement.

**Bias-Variance Tradeoff**  
Each individual predictor has a higher bias than if it were trained on the entire original dataset. However aggregating them mathematically reduces both bias and variance. Bagging is typically preferred in production because the extra diversity reduces the ensemble's variance even further.

In practice, the ensemble often ends up with a similar bias but a lower variance than a single predictor trained on the original training set. Therefore it works best with high-variance and low-bias models (e.g., decision trees).

### Bagging and Pasting in Scikit-Learn

Scikit-Learn provides a simple API for both bagging and pasting through:

- `BaggingClassifier` (for classification)
- `BaggingRegressor` (for regression)

The following example trains an ensemble of 500 Decision Tree classifiers, each trained on 100 training instances.

In [11]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

bag_clf = BaggingClassifier(DecisionTreeClassifier(), n_estimators=500,
                            max_samples=100, n_jobs=-1, random_state=42)
bag_clf.fit(X_train, y_train)

print('Bagging Ensemble Accuracy \n', bag_clf.score(X_test, y_test))

Bagging Ensemble Accuracy 
 0.904


This is an example of bagging, but if you want to use pasting instead, just set `bootstrap=False`.

**Note:** `BaggingClassifier` automatically performs soft voting when the base classifier supports `predict_proba()`.

### Out of Bag Evaluation

When utilizing Bagging with replacement mathematical probability dictates that only about 63% of the training instances are uniquely sampled for any given predictor. The untouched 37% of instances are structurally isolated and called Out of Bag instances.

Because these instances were never seen by the predictor during training they can be evaluated to calculate a highly accurate and unbiased validation score without actually sacrificing any valuable data to create a separate validation set.

In Scikit-Learn, you can set `oob_score=True` when creating a `BaggingClassifier` to request an automatic OOB evaluation after training. The resulting evaluation score is available in the `oob_score_` attribute.

In [14]:
bag_clf_oob = BaggingClassifier(DecisionTreeClassifier(), n_estimators=500,
                            oob_score=True, n_jobs=-1, random_state=42)
bag_clf_oob.fit(X_train, y_train)
print('Out of Bag Evaluation Score \n', bag_clf_oob.oob_score_)

Out of Bag Evaluation Score 
 0.896


In [15]:
from sklearn.metrics import accuracy_score
y_pred = bag_clf_oob.predict(X_test)
print('Accuracy on the test set \n', accuracy_score(y_test, y_pred))

Accuracy on the test set 
 0.92


### Random Patches and Random Subspaces

The BaggingClassifier natively supports sampling features alongside instances. This is exceptionally useful for high dimensional inputs like images. 
* **Random Patches method**: Sampling both instances and features  
* **Random Subspaces method**: Keeping all training instances but sampling features

Sampling features results in even more predictor diversity, trading a bit more bias for a lower variance.

## Random Forests

Random Forest is one of the most widely used ensemble learning algorithms.

It can be viewed as a Bagging ensemble of Decision Trees with an additional layer of randomness. During training, each split considers only a random subset of features rather than evaluating every available feature.

This extra randomness reduces correlation among trees, leading to lower variance and better generalization performance.

Compared with a standard Bagging ensemble of Decision Trees, Random Forests often achieve similar or slightly better predictive performance while requiring less hyperparameter tuning.

In [18]:
from sklearn.ensemble import RandomForestClassifier

rnd_clf = RandomForestClassifier(n_estimators=500, max_leaf_nodes=16,
                                 n_jobs=-1, random_state=42)
rnd_clf.fit(X_train, y_train)

y_pred_rf = rnd_clf.predict(X_test)

print("Random Forest Accuracy\n", rnd_clf.score(X_test, y_test))

Random Forest Accuracy
 0.912


The Random Forest classifier achieves competitive performance while requiring minimal preprocessing and hyperparameter tuning.

The additional feature randomness introduced at each split helps decorrelate individual trees, making the ensemble more robust than a simple collection of highly similar Decision Trees.

### Extra-Trees

Extra-Trees (Extremely Randomized Trees) introduce even more randomness than Random Forests.

While Random Forests search for the best split threshold among a random subset of features, Extra-Trees select split thresholds randomly.

This increases bias slightly but can further reduce variance and significantly speed up training.

You can create an extra-trees classifier/regressor using Scikit-Learn’s `ExtraTreesClassifier`/`ExtraTreesRegressor` class.

### Feature importance

Feature importance measures how useful each feature is for making predictions across the ensemble.

Features that frequently contribute to high-quality splits receive higher importance scores.

This provides a simple form of model interpretability and can help identify which variables have the strongest influence on predictions.

In [22]:
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)

rnd_clf = RandomForestClassifier(n_estimators=500, random_state=42)
rnd_clf.fit(iris.data, iris.target)

for score, name in zip(rnd_clf.feature_importances_, iris.data.columns):
    print(round(score, 2), name)

0.11 sepal length (cm)
0.02 sepal width (cm)
0.44 petal length (cm)
0.42 petal width (cm)


## Boosting

Boosting is another ensemble learning technique that combines multiple weak learners into a strong learner.

Unlike bagging, where predictors are trained independently and in parallel, boosting trains predictors sequentially. Each new predictor attempts to correct the errors made by the previous ensemble.

As a result, boosting primarily reduces bias, whereas bagging primarily reduces variance.

Boosting methods are among the most powerful machine learning algorithms for structured tabular datasets.

### AdaBoost

AdaBoost, short for Adaptive Boosting, was one of the first successful boosting algorithms.

The algorithm trains a sequence of weak learners, usually shallow decision trees called decision stumps. After each learner is trained, the weights of misclassified training instances are increased. This forces the next learner to focus more heavily on the difficult examples.

The process repeats until the desired number of predictors has been trained.

In [45]:
from sklearn.ensemble import AdaBoostClassifier

ada_clf = AdaBoostClassifier(
    DecisionTreeClassifier(max_depth=1), n_estimators=30,
    learning_rate=0.5, random_state=42, algorithm="SAMME")
ada_clf.fit(X_train, y_train)

print('AdaBoost Accuracy\n', ada_clf.score(X_test, y_test))

AdaBoost Accuracy
 0.88


AdaBoost combines many simple decision stumps into a much stronger classifier.

Each new stump focuses more heavily on the instances that were previously misclassified. Over time, the ensemble gradually improves its performance by correcting its own mistakes.

The `learning_rate` hyperparameter controls how much influence each predictor has on the final ensemble.